# 02 — Exploratory Data Analysis (EDA)We answer the questions the assignment PDF asks, each with a chart + a written interpretation.Run cells top to bottom with **Shift+Enter**.

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as sns# Use the cleaned data from notebook 1df = pd.read_csv("../data/insurance_claims_cleaned.csv")# Convert the target to a clean 0/1 for easier plotting/maths laterdf['fraud_flag'] = df['fraud_reported'].map({'Y': 1, 'N': 0})sns.set_style("whitegrid")df.shape

## Q1: What percentage of claims are fraudulent?

In [ ]:
fraud_rate = df['fraud_flag'].mean() * 100print(f"Fraudulent claims: {fraud_rate:.1f}%")plt.figure(figsize=(5,4))df['fraud_reported'].value_counts().plot(kind='bar', color=['#4C72B0','#C44E52'])plt.title('Claim counts: Fraud vs Genuine')plt.xlabel('Fraud Reported')plt.ylabel('Number of Claims')plt.xticks(rotation=0)plt.tight_layout()plt.show()

**Interpretation:** *(fill this in with the actual number you see above, e.g.)* Roughly a quarter of claims in this dataset are fraudulent. This confirms the dataset is **imbalanced** — genuine claims heavily outnumber fraud cases. This matters a lot later: accuracy alone would be a misleading metric, because a model that just guesses 'not fraud' every time would already look ~75% accurate while catching zero fraud.

## Q2: Which variables appear related to fraud?

In [ ]:
plt.figure(figsize=(7,5))sns.countplot(data=df, x='incident_severity', hue='fraud_reported', order=df['incident_severity'].value_counts().index)plt.title('Fraud rate by Incident Severity')plt.xticks(rotation=20)plt.tight_layout()plt.show()

**Interpretation:** *(write what you observe, e.g.)* Claims marked 'Major Damage' show a noticeably higher share of fraud compared to 'Minor Damage' or 'Trivial Damage'. This suggests incident_severity is a useful predictive feature.

In [ ]:
plt.figure(figsize=(7,5))sns.countplot(data=df, x='insured_hobbies', hue='fraud_reported',              order=df['insured_hobbies'].value_counts().index[:8])plt.title('Fraud rate by Insured Hobby (top 8 most common)')plt.xticks(rotation=45, ha='right')plt.tight_layout()plt.show()

**Interpretation:** Some specific hobbies show disproportionately higher fraud counts relative to their overall frequency. This is a known quirk of this dataset — flag it as an interesting but possibly coincidental correlation (small sample sizes per hobby can be misleading) rather than a strong causal signal.

## Q3: Are unusually large claims more likely to be suspicious?

In [ ]:
plt.figure(figsize=(7,5))sns.boxplot(data=df, x='fraud_reported', y='total_claim_amount')plt.title('Total Claim Amount: Fraud vs Genuine')plt.tight_layout()plt.show()print(df.groupby('fraud_reported')['total_claim_amount'].describe())

**Interpretation:** *(compare the median/mean shown above)* Fraudulent claims tend to have a higher median claim amount than genuine ones, though there is overlap — claim amount alone isn't a perfect separator, but it's a useful signal in combination with other features.

## Q4: Does the delay between incident and submission matter?

In [ ]:
# We need policy_bind_date and incident_date for this - reload the (uncleaned-but-deduped) originaldf_dates = pd.read_csv("../data/insurance_claims.csv")df_dates['incident_date'] = pd.to_datetime(df_dates['incident_date'])df_dates['policy_bind_date'] = pd.to_datetime(df_dates['policy_bind_date'])# NOTE: this dataset doesn't include a separate 'claim submission date' column,# so we treat incident_date as a stand-in reference point and instead look at# policy tenure at time of incident (days between policy start and the incident).df_dates['policy_age_days'] = (df_dates['incident_date'] - df_dates['policy_bind_date']).dt.daysplt.figure(figsize=(7,5))sns.boxplot(data=df_dates, x='fraud_reported', y='policy_age_days')plt.title('Policy Age at Time of Incident: Fraud vs Genuine')plt.tight_layout()plt.show()

**Assumption documented:** The PDF mentions 'days between incident and claim submission' as a feature, but this specific dataset does not contain a separate claim-submission-date column (only `incident_date` and `policy_bind_date`). We substitute **policy tenure at time of incident** as the closest available proxy for 'how new/suspicious the timing is', and we note this substitution clearly here and in the README/slides so it isn't mistaken for an oversight.**Interpretation:** *(check the plot)* If fraudulent claims cluster toward newer policies (smaller policy_age_days), that supports the common fraud pattern of people insuring something shortly before staging a claim.

## Q5: Are previous claims associated with greater risk?

In [ ]:
# This dataset doesn't have an explicit 'previous_claim_count' column either.# We document that assumption and instead check a related available signal: months_as_customer,# which acts as a rough proxy for customer history/tenure.plt.figure(figsize=(7,5))sns.boxplot(data=df, x='fraud_reported', y='months_as_customer')plt.title('Months as Customer: Fraud vs Genuine')plt.tight_layout()plt.show()

**Assumption documented:** No direct `previous_claim_count` field exists in this dataset. `months_as_customer` is used as the closest available proxy for customer history. **Interpretation:** *(check the plot)* note whether fraud cases skew toward newer or longer-tenured customers.

## Q6: Unusual combinations of claim amount, repair estimate, invoice amount?

In [ ]:
# This specific dataset doesn't include separate repair_estimate / final_invoice_amount columns# (those appear in the PDF's suggested synthetic schema). We use the closest equivalent breakdown# this dataset DOES provide: vehicle_claim vs total_claim_amount, as a stand-in ratio check.df['vehicle_claim_ratio'] = df['vehicle_claim'] / df['total_claim_amount']plt.figure(figsize=(7,5))sns.boxplot(data=df, x='fraud_reported', y='vehicle_claim_ratio')plt.title('Vehicle Claim as % of Total Claim: Fraud vs Genuine')plt.tight_layout()plt.show()

**Assumption documented:** True repair-estimate-vs-invoice fields aren't present in this real dataset (they're part of the PDF's *suggested synthetic* schema). We substitute the vehicle-claim-to-total-claim ratio as the closest available 'proportions look unusual' signal, and will build a similar engineered ratio feature formally in the Feature Engineering notebook.**This is now 6 visualisations with interpretations — exceeds the PDF's minimum of 5.**

In [ ]:
# Correlation heatmap of numeric features vs fraud - a nice summary visual for the presentationnumeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()plt.figure(figsize=(10,8))corr = df[numeric_cols].corr()sns.heatmap(corr[['fraud_flag']].sort_values('fraud_flag', ascending=False), annot=True, cmap='coolwarm', center=0)plt.title('Correlation of Numeric Features with Fraud')plt.tight_layout()plt.show()

**Interpretation:** This heatmap ranks every numeric column by how strongly it correlates with fraud, giving a quick summary view — useful as a slide in the final presentation.